# 05 — Pipeline Verification & Validation

End-to-end verification using synthetic data with known ground truth.
Validates that all modules work correctly and produce expected outputs.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data.fetcher import load_config
from src.data.preprocessor import (
    align_panel, impute_missing, add_pstn_phaseout_feature, 
    add_derived_features, build_product_lifetime_table, save_processed
)
from src.models.logistic_growth import (
    logistic, fit_country, fit_all_countries, predict_years, summarize_results
)
from src.models.survival import (
    fit_cox_model, predict_survival_probability, rank_markets
)

print("All modules imported successfully.")

## 5.1 Generate Synthetic Data with Known Ground Truth

In [ ]:
np.random.seed(42)
config = load_config()

countries_list = []
for region in config['countries'].values():
    for item in region:
        for code, name in item.items():
            countries_list.append((code, name))

# Known ground truth parameters for each country
ground_truth = {}
rows = []
for i, (code, name) in enumerate(countries_list):
    true_K = np.random.uniform(30, 70)
    true_r = np.random.uniform(0.15, 0.40)
    true_t0 = np.random.uniform(2003, 2015)
    ground_truth[code] = {'K': true_K, 'r': true_r, 't0': true_t0}
    
    for year in range(2000, 2026):
        val = logistic(np.array([float(year)]), true_K, true_r, true_t0)[0]
        val += np.random.normal(0, 1.0)  # noise
        broadband = min(45, (year - 2000) * 2.0 + np.random.normal(0, 2))
        gdp = 20000 + (year - 2000) * 600 + np.random.normal(0, 500)
        urban = 60 + (year - 2000) * 0.3 + np.random.normal(0, 1)
        rows.append({
            'country': code,
            'year': year,
            'fixed_subs_value': max(0, val),
            'broadband_value': max(0, broadband),
            'gdp_per_capita_value': max(0, gdp),
            'urban_pop_value': min(100, max(0, urban)),
        })

panel = pd.DataFrame(rows)
panel = add_pstn_phaseout_feature(panel, config['data']['pstn_switchoff'])
print(f"Synthetic panel: {panel.shape[0]} rows, {panel.shape[1]} cols")
print(f"Ground truth parameters for {len(ground_truth)} countries")

---
## 5.2 Test 1: Data Processing Pipeline

In [ ]:
t1_pass = True

aligned = align_panel({'synthetic': panel}, panel['country'].unique(), range(2000, 2026))
t1_pass &= len(aligned) == len(panel)
print(f"[TEST] align_panel preserves all rows: {len(aligned)} == {len(panel)} → {'PASS' if t1_pass else 'FAIL'}")

imputed = impute_missing(aligned, method='linear', max_gap=3)
# Imputation should leave no NaNs in numeric columns (bfill/ffill backstop).
num_cols = imputed.select_dtypes(include='number').columns
t1_pass &= imputed[num_cols].isnull().sum().sum() == 0
print(f"[TEST] impute_missing completes: {imputed.shape} → {'PASS' if t1_pass else 'FAIL'}")

derived = add_derived_features(aligned)
t1_pass &= 'fixed_subs_value_yoy' in derived.columns
print(f"[TEST] add_derived_features adds yoy: {'PASS' if t1_pass else 'FAIL'}")

has_pstn = 'has_pstn_phaseout' in panel.columns
t1_pass &= has_pstn
print(f"[TEST] add_pstn_phaseout_feature: {'PASS' if has_pstn else 'FAIL'}")

print(f"\n>>> Data Pipeline: {'ALL PASS' if t1_pass else 'SOME FAILED'} <<<")

## 5.3 Test 2: Logistic Growth Model Recovery

In [ ]:
t2_pass = True

results = fit_all_countries(panel, death_threshold=0.05)
summary = summarize_results(results)

n_converged = summary['converged'].sum()
t2_pass &= n_converged > 0
print(f"[TEST] Countries converged: {n_converged}/{len(summary)} → {'PASS' if t2_pass else 'FAIL'}")

# Check that K values are recovered reasonably
for code in ['tw', 'jp', 'us']:
    row = results[results['country'] == code]
    if not row.empty and row.iloc[0]['converged']:
        true = ground_truth.get(code, {})
        if true:
            err = abs(row.iloc[0]['K'] - true['K']) / true['K']
            ok = err < 0.5  # Within 50%
            t2_pass &= ok
            print(f"[TEST] {code}: K_recovered={row.iloc[0]['K']:.1f}, K_true={true['K']:.1f}, error={err:.1%} → {'PASS' if ok else 'FAIL'}")

print(f"\n>>> Logistic Model: {'ALL PASS' if t2_pass else 'SOME FAILED'} <<<")

## 5.4 Test 3: Survival Model

In [ ]:
t3_pass = True

# The Test-2 growth panel above is monotonically increasing (pure S-curve) with
# covariates that have NO cross-country variation, so it yields zero "death"
# events and a singular Cox design. To verify the survival model meaningfully we
# build a dedicated synthetic panel with (a) a realistic rise-then-decline
# lifecycle so deaths actually occur, and (b) per-country covariate variation,
# with higher early broadband driving a faster decline (a known ground-truth
# relationship the Cox model should recover as a hazard ratio > 1).
np.random.seed(7)
surv_codes = [c for region in config['countries'].values() for item in region for c in item]
surv_rows = []
for code in surv_codes:
    bb = np.random.uniform(3, 30)
    gdp = np.random.uniform(15000, 45000)
    urb = np.random.uniform(55, 90)
    peak = np.random.uniform(40, 70)
    peak_year = np.random.uniform(2006, 2012)
    decline_rate = 0.10 + 0.012 * bb + np.random.uniform(0, 0.05)  # broadband speeds death
    for year in range(2000, 2026):
        if year <= peak_year:
            val = peak / (1 + np.exp(-0.4 * (year - (peak_year - 4))))
        else:
            val = peak * np.exp(-decline_rate * (year - peak_year))
        surv_rows.append({
            'country': code, 'year': year,
            'fixed_subs_value': max(0, val + np.random.normal(0, 0.5)),
            'broadband_value': bb + np.random.normal(0, 0.3),
            'gdp_per_capita_value': gdp + np.random.normal(0, 200),
            'urban_pop_value': urb + np.random.normal(0, 0.3),
        })
surv_panel = pd.DataFrame(surv_rows)
surv_panel = add_pstn_phaseout_feature(surv_panel, config['data']['pstn_switchoff'])

lifetime = build_product_lifetime_table(
    surv_panel, product_intro_year=2005,
    penetration_col='fixed_subs_value',
)
t3_pass &= len(lifetime) > 0
n_events = int(lifetime['product_dead'].sum())
t3_pass &= n_events >= 2  # survival model is only identifiable with >=2 events
print(f"[TEST] Lifetime table built: {len(lifetime)} countries, {n_events} deaths → {'PASS' if t3_pass else 'FAIL'}")

available_covs = ['broadband_value', 'gdp_per_capita_value', 'has_pstn_phaseout']
available_covs = [c for c in available_covs if c in lifetime.columns]

model = fit_cox_model(lifetime, covariates=available_covs, penalizer=0.1)
t3_pass &= hasattr(model, 'hazard_ratios_')
print(f"[TEST] CoxPH model fitted: {len(model.hazard_ratios_)} covariates → {'PASS' if t3_pass else 'FAIL'}")

rankings = rank_markets(model, lifetime, t=5)
t3_pass &= len(rankings) == len(lifetime)
t3_pass &= rankings['survival_prob_5y'].is_monotonic_decreasing
print(f"[TEST] Rankings sorted correctly: {len(rankings)} entries → {'PASS' if t3_pass else 'FAIL'}")

probs = []
for _, row in rankings.head(3).iterrows():
    covs = {c: lifetime[lifetime['country'] == row['country']][c].iloc[0] for c in available_covs}
    p = predict_survival_probability(model, covs, t=5)
    probs.append(p)
t3_pass &= all(0 <= p <= 1 for p in probs)
print(f"[TEST] Survival probabilities in [0,1]: {[f'{p:.2f}' for p in probs]} → {'PASS' if t3_pass else 'FAIL'}")

print(f"\n>>> Survival Model: {'ALL PASS' if t3_pass else 'SOME FAILED'} <<<")

## 5.5 Test 4: Config Completeness

In [ ]:
t4_pass = True

required_keys = ['countries', 'model', 'data']
for k in required_keys:
    t4_pass &= k in config
print(f"[TEST] Required config keys: {required_keys} → {'PASS' if t4_pass else 'FAIL'}")

required_model_keys = ['logistic_growth', 'survival']
for k in required_model_keys:
    t4_pass &= k in config.get('model', {})
print(f"[TEST] Model config sections: {required_model_keys} → {'PASS' if t4_pass else 'FAIL'}")

death_config = config.get('model', {}).get('logistic_growth', {}).get('death_threshold')
t4_pass &= death_config == 0.3
print(f"[TEST] Death threshold = 0.3: {death_config} → {'PASS' if t4_pass else 'FAIL'}")

print(f"\n>>> Config Validation: {'ALL PASS' if t4_pass else 'SOME FAILED'} <<<")

---
## 5.6 Overall Verification Result

In [ ]:
total_pass = t1_pass and t2_pass and t3_pass and t4_pass
print("=" * 50)
print("  PIPELINE VERIFICATION RESULTS")
print("=" * 50)
print(f"  1. Data Processing:       {'PASS' if t1_pass else 'FAIL'}")
print(f"  2. Logistic Growth Model: {'PASS' if t2_pass else 'FAIL'}")
print(f"  3. Survival Model:        {'PASS' if t3_pass else 'FAIL'}")
print(f"  4. Config Validation:     {'PASS' if t4_pass else 'FAIL'}")
print("-" * 50)
print(f"  OVERALL: {'ALL TESTS PASSED' if total_pass else 'SOME TESTS FAILED'}")
print("=" * 50)

**EN — What this verifies.** Each test recovers known ground truth from synthetic data: the data pipeline preserves rows and imputes gaps, the logistic model recovers planted parameters within tolerance, the survival model produces valid in-[0,1] monotone survival probabilities, and the config is complete. 'ALL TESTS PASSED' means the end-to-end pipeline is internally consistent — it does not validate the real-world data itself.

**繁中 — 此驗證的意義。** 每項測試以合成資料還原已知真值：資料管線保留列數並補值、邏輯斯模型在容差內還原植入參數、存活模型輸出落在 [0,1] 且單調的存活機率、設定檔完整。'ALL TESTS PASSED' 表示端到端管線內部一致——但不代表已驗證真實世界資料本身的正確性。

In [ ]:
# Save verification results summary
summary_df = pd.DataFrame([{
    'test': 'data_pipeline', 'passed': t1_pass,
}, {
    'test': 'logistic_model', 'passed': t2_pass,
}, {
    'test': 'survival_model', 'passed': t3_pass,
}, {
    'test': 'config_validation', 'passed': t4_pass,
}, {
    'test': 'overall', 'passed': total_pass,
}])
summary_df.to_csv('data/processed/verification_results.csv', index=False)
print("Verification results saved to data/processed/verification_results.csv")